# 08 · Consolidación de resultados
Reúne en una sola tabla los experimentos de Big Data (guardados automáticamente) y los de BI (B1 y B2 ya vienen cargados con los tiempos medidos en SQL Server).
Con esta tabla se arma el dashboard comparativo y las tablas del capítulo VI del informe.

In [ ]:
%run ./00_utilidades

## 1. Resultados de BI (SQL Server)
Los tiempos de B1 y B2 ya están cargados (vienen de `log.EXPERIMENTOS`). Si repites alguna medición, actualiza las cifras.

In [ ]:
BI = [  # (experimento, variante, escala, filas, [tiempos en s], entorno, responsable, detalle)
    ("B1_etl_volumen",   "sql_etl_1x",        "1x", 1_048_575, [44.675, 41.511, 40.761],  "SQL Server", "Huaynate",         "ETL (stg a dm)"),
    ("B1_etl_volumen",   "sql_etl_2x",        "2x", 2_048_575, [99.871, 95.372, 92.347],  "SQL Server", "Huaynate",         "ETL (stg a dm)"),
    ("B1_etl_volumen",   "sql_etl_5x",        "5x", 5_048_575, [248.525, 249.849, 239.417], "SQL Server", "Huaynate",       "ETL (stg a dm)"),
    ("B2_indices_hechos","sin_indices",       "5x", 5_048_575, [5.306, 4.842, 4.845],     "SQL Server", "Andrade y Quispe", "consulta de KPIs"),
    ("B2_indices_hechos","indice_cubriente",  "5x", 5_048_575, [3.134, 3.287, 3.284],     "SQL Server", "Andrade y Quispe", "consulta de KPIs"),
    ("B2_indices_hechos","columnstore",       "5x", 5_048_575, [0.387, 0.385, 0.384],     "SQL Server", "Andrade y Quispe", "consulta de KPIs"),
]
pendientes = 0
for exp, var, esc, filas, ts, entorno, resp, det in BI:
    if all(t == 0.0 for t in ts):
        pendientes += 1
        continue
    registrar_resultado(exp, var, esc, filas, ts, entorno, resp, det)
print(f"Registrados: {len(BI) - pendientes} | pendientes (con 0.0): {pendientes}")

## 2. Tabla consolidada

In [ ]:
res = leer_tabla(T_RESULTADOS)
display(res.orderBy("experimento", "variante"))

## 3. Gráficos para el informe
1. Ejecuta la celda de abajo y, en la salida, usa **+ > Visualization**: tipo *Bar*, eje X `variante`, eje Y `media_s`, agrupa por `entorno`.
2. Toma una captura por experimento (filtra con `WHERE experimento = '...'`).
3. Para un dashboard, en **Dashboards** agrega el dataset `SELECT * FROM workspace.experimentobigdata.resultados_experimentos` y crea barras de `media_s` por `variante`.

In [ ]:
display(res.select("experimento", "variante", "escala", "entorno", "responsable", "media_s", "desv_s", "throughput_filas_s", "detalle")
          .orderBy("experimento", "media_s"))

In [ ]:
# Comparación general: tiempo medio por experimento y entorno
display(res.groupBy("experimento", "entorno").agg(F.round(F.avg("media_s"), 3).alias("media_promedio_s"),
                                                  F.count("*").alias("variantes")).orderBy("experimento"))